# Task 4 (Optional): Prioritize Potential Drug Target Genes

Ranking the 12 ALS genes by therapeutic potential using three criteria:
1. **Disease reversal** — does knockout shift disease cells toward healthy? (using corrected per-cell-type scores)
2. **Cell-type specificity** — stronger effect in vulnerable neurons = better target
3. **Dose sensitivity** — genes where even mild perturbation has an effect are higher priority (lower dose = fewer off-target effects)

Then I cross-check against known ALS therapeutics (SOD1/tofersen, ATXN2/ASOs, etc.) and the DE ranking.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import sparse, stats
import pickle

sns.set_theme(style='whitegrid')

## 1. Load all results

In [ ]:
adata = sc.read_h5ad('../data/counts_combined_filtered_BA4_sALS_PN.h5ad')
emb_baseline = np.load('../data/embeddings_baseline.npy')
d2h_direction = np.load('../data/disease_to_healthy_direction.npy')
df_reversal = pd.read_csv('../data/reversal_scores.csv')

with open('../data/results_disease.pkl', 'rb') as f:
    results_disease = pickle.load(f)

DISEASE_COL   = 'Condition'
CELLTYPE_COL  = 'CellType'
SUBTYPE_COL   = 'SubType'
HEALTHY_LABEL = 'PN'
DISEASE_LABEL = 'ALS'

healthy_mask = (adata.obs[DISEASE_COL] == HEALTHY_LABEL).values
disease_mask = (adata.obs[DISEASE_COL] == DISEASE_LABEL).values

genes = list(results_disease.keys())
print(f"Healthy ({HEALTHY_LABEL}): {healthy_mask.sum():,}, Disease ({DISEASE_LABEL}): {disease_mask.sum():,}")
print(f"Genes to rank: {genes}")

# Load corrected reversal scores if available
CORRECTED_PATH = '../data/corrected_reversal_scores.csv'
if os.path.exists(CORRECTED_PATH):
    df_corrected = pd.read_csv(CORRECTED_PATH)
    print(f"Loaded corrected reversal scores: {len(df_corrected)} rows")
    USE_CORRECTED = True
else:
    print("No corrected reversal scores found — using global reversal")
    USE_CORRECTED = False

## 2. Criterion 1: Disease reversal score

How much does knocking down this gene shift disease cells toward the healthy centroid? We use the knockout (factor=0) results as the strongest therapeutic analog.

In [ ]:
# Reversal score — use per-cell-type corrected scores if available
score_reversal = {}

if USE_CORRECTED:
    print("Using PER-CELL-TYPE corrected reversal scores (composition confound removed)")
    for gene in genes:
        ko_row = df_corrected[(df_corrected['gene'] == gene) & (df_corrected['dose'] == 'knockout')]
        if len(ko_row) > 0:
            score_reversal[gene] = {
                'mean_reversal': ko_row['corrected_reversal'].values[0],
                'global_reversal': ko_row['global_reversal'].values[0],
            }
else:
    print("Using global reversal scores (uncorrected)")
    centroid_healthy = np.load('../data/centroid_healthy.npy')
    centroid_disease = np.load('../data/centroid_disease.npy')
    d2h = centroid_healthy - centroid_disease
    d2h_norm = d2h / (np.linalg.norm(d2h) + 1e-8)
    for gene in genes:
        if 'knockout' in results_disease[gene]:
            mean_shift = results_disease[gene]['knockout'].get('mean_shift_vector')
            if mean_shift is not None:
                score_reversal[gene] = {
                    'mean_reversal': np.dot(mean_shift, d2h_norm),
                }

df_c1 = pd.DataFrame(score_reversal).T
df_c1.index.name = 'gene'
print("\nCriterion 1 — Disease reversal:")
print(df_c1.sort_values('mean_reversal', ascending=False).to_string())

## 3. Criterion 2: Cell-type specificity

An ideal drug target affects vulnerable cell types (Betz cells / projection neurons) more than others. We compute a specificity ratio: shift in vulnerable cells / shift in all cells.

In [ ]:
score_specificity = {}

if CELLTYPE_COL:
    disease_celltypes = adata[disease_mask].obs[CELLTYPE_COL].values
    n_perturbed = results_disease[genes[0]]['knockout']['cosine_distance'].shape[0]
    if n_perturbed < disease_mask.sum():
        rng = np.random.RandomState(42)
        idx = rng.choice(disease_mask.sum(), n_perturbed, replace=False)
        disease_celltypes = disease_celltypes[idx]
    
    # Identify candidate vulnerable cell types
    # Betz cells, projection neurons, excitatory neurons
    ct_values = pd.Series(disease_celltypes)
    vulnerable_keywords = ['betz', 'l5', 'layer 5', 'projection', 'excitatory', 'motor', 'pyramidal', 'ET ']
    vulnerable_cts = [ct for ct in ct_values.unique() if any(kw in str(ct).lower() for kw in vulnerable_keywords)]
    
    if not vulnerable_cts:
        # Fallback: take the top cell type by count
        top_ct = ct_values.value_counts().index[0]
        vulnerable_cts = [top_ct]
        print(f"No obvious vulnerable cell types found. Using '{top_ct}' as reference.")
    else:
        print(f"Vulnerable cell types identified: {vulnerable_cts}")
    
    vulnerable_mask_local = ct_values.isin(vulnerable_cts).values
    other_mask_local = ~vulnerable_mask_local
    
    for gene in genes:
        if 'knockout' not in results_disease[gene]:
            continue
        cos_dist = results_disease[gene]['knockout']['cosine_distance']
        shift_vulnerable = cos_dist[vulnerable_mask_local].mean() if vulnerable_mask_local.sum() > 0 else 0
        shift_other = cos_dist[other_mask_local].mean() if other_mask_local.sum() > 0 else 1e-8
        
        score_specificity[gene] = {
            'shift_vulnerable': shift_vulnerable,
            'shift_other': shift_other,
            'specificity_ratio': shift_vulnerable / (shift_other + 1e-8),
        }

df_c2 = pd.DataFrame(score_specificity).T
df_c2.index.name = 'gene'
print("\nCriterion 2 — Cell-type specificity:")
print(df_c2.sort_values('specificity_ratio', ascending=False).to_string())

## 4. Criterion 3: Dose sensitivity

Genes where even mild perturbations (factor=0.5 or 1.5) cause meaningful shifts are higher-priority targets — a milder intervention means fewer off-target effects.

In [ ]:
score_dose_sensitivity = {}

for gene in genes:
    # Collect all dose points
    factors = []
    shifts = []
    for dose_name, effect in results_disease[gene].items():
        f = effect['factor']
        if f == 1.0:
            continue  # skip baseline
        factors.append(np.log2(f + 1e-8))  # log-scale for linear regression
        shifts.append(effect['cosine_distance'].mean())
    
    if len(factors) >= 3:
        # Slope of shift vs log2(factor) — steeper = more sensitive
        slope, intercept, r_value, p_value, std_err = stats.linregress(factors, shifts)
        
        # Also check the mild perturbation effect specifically
        mild_shift = 0
        for dose_name in ['moderate_down', 'mild_up']:
            if dose_name in results_disease[gene]:
                mild_shift = max(mild_shift, results_disease[gene][dose_name]['cosine_distance'].mean())
        
        score_dose_sensitivity[gene] = {
            'dose_slope': abs(slope),
            'r_squared': r_value**2,
            'mild_perturbation_shift': mild_shift,
        }

df_c3 = pd.DataFrame(score_dose_sensitivity).T
df_c3.index.name = 'gene'
print("Criterion 3 — Dose sensitivity:")
print(df_c3.sort_values('mild_perturbation_shift', ascending=False).to_string())

## 5. Composite ranking

In [ ]:
# Combine criteria into a single ranking
df_rank = pd.DataFrame(index=genes)

# C1: reversal (higher = better)
if len(df_c1) > 0:
    df_rank['reversal_score'] = df_c1['mean_reversal']
    df_rank['reversal_rank'] = df_rank['reversal_score'].rank(ascending=False)

# C2: specificity (higher ratio = better)
if len(df_c2) > 0:
    df_rank['specificity_ratio'] = df_c2['specificity_ratio']
    df_rank['specificity_rank'] = df_rank['specificity_ratio'].rank(ascending=False)

# C3: dose sensitivity (higher mild shift = better)
if len(df_c3) > 0:
    df_rank['mild_shift'] = df_c3['mild_perturbation_shift']
    df_rank['dose_rank'] = df_rank['mild_shift'].rank(ascending=False)

# Composite: average rank (lower = better target)
rank_cols = [c for c in df_rank.columns if c.endswith('_rank')]
df_rank['composite_rank'] = df_rank[rank_cols].mean(axis=1)
df_rank = df_rank.sort_values('composite_rank')

print("=" * 80)
print("DRUG TARGET GENE PRIORITIZATION")
print("=" * 80)
print(df_rank.to_string())

In [ ]:
# Visualization: multi-criteria scatter
fig, ax = plt.subplots(figsize=(10, 8))

x = df_rank['reversal_score'] if 'reversal_score' in df_rank else np.zeros(len(df_rank))
y = df_rank['mild_shift'] if 'mild_shift' in df_rank else np.zeros(len(df_rank))
size = df_rank['specificity_ratio'] * 200 if 'specificity_ratio' in df_rank else np.full(len(df_rank), 100)
# Clamp sizes for display
size = np.clip(size, 50, 500)

scatter = ax.scatter(x, y, s=size, c=df_rank['composite_rank'], cmap='RdYlGn_r', edgecolors='black', alpha=0.8)

for i, gene in enumerate(df_rank.index):
    ax.annotate(gene, (x.iloc[i], y.iloc[i]), fontsize=10, fontweight='bold',
                ha='center', va='bottom', xytext=(0, 8), textcoords='offset points')

ax.set_xlabel('Disease reversal score (knockout)', fontsize=12)
ax.set_ylabel('Dose sensitivity (mild perturbation shift)', fontsize=12)
ax.set_title('Drug target prioritization\n(size = cell-type specificity, color = composite rank)', fontsize=13)
ax.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
plt.colorbar(scatter, label='Composite rank (lower = better)')
plt.tight_layout()
plt.savefig('../slides/task4_target_prioritization.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Validation against known ALS therapeutic targets

In [ ]:
# Known therapeutic targets with clinical evidence
known_targets = {
    'SOD1': 'Tofersen (ASO) — FDA approved 2023 for SOD1-ALS',
    'ATXN2': 'BIIB105/ION541 (ASO) — Phase 1/2 trial',
    'STMN2': 'Downstream of TDP-43; ASO strategies under development',
    'UNC13A': 'Cryptic exon target of TDP-43; ASO in preclinical',
    'C9orf72': 'Multiple ASOs in clinical trials (BIIB078, WVE-004)',
    'FUS': 'ION363 (ASO) — compassionate use',
}

print("Validation: Our ranking vs known ALS therapeutic targets")
print("=" * 70)
for i, gene in enumerate(df_rank.index):
    status = known_targets.get(gene, 'No current clinical program')
    rank = i + 1
    rev = df_rank.loc[gene, 'reversal_score'] if 'reversal_score' in df_rank else float('nan')
    print(f"  #{rank:2d} {gene:10s}  reversal={rev:+.4f}  | {status}")

In [ ]:
# Compare with DE-derived gene ranking
# The "naive" approach: rank genes purely by differential expression
adata.obs['_condition'] = adata.obs[DISEASE_COL].astype(str)
sc.tl.rank_genes_groups(adata, groupby='_condition', groups=[DISEASE_LABEL], reference=HEALTHY_LABEL, method='wilcoxon')
de_results = sc.get.rank_genes_groups_df(adata, group=DISEASE_LABEL)

# Where do our target genes fall in the DE ranking?
de_ranks = {}
for gene in genes:
    match = de_results[de_results['names'] == gene]
    if len(match) > 0:
        de_rank_val = match.index[0] + 1
        lfc = match['logfoldchanges'].values[0]
        pval = match['pvals_adj'].values[0]
        de_ranks[gene] = {'de_rank': de_rank_val, 'log2fc': lfc, 'padj': pval}

df_de_validation = pd.DataFrame(de_ranks).T

print("\nOur target genes in the DE ranking:")
print(f"(Total genes tested: {len(de_results)})")
for gene in df_rank.index:
    if gene in df_de_validation.index:
        row = df_de_validation.loc[gene]
        sig = '***' if row['padj'] < 0.001 else '**' if row['padj'] < 0.01 else '*' if row['padj'] < 0.05 else 'ns'
        print(f"  {gene:10s}  DE rank: {int(row['de_rank']):>5d}  log2FC: {row['log2fc']:+.3f}  padj: {row['padj']:.2e}  {sig}")
    else:
        print(f"  {gene:10s}  not in DE results")

## 7. Final target list

In [ ]:
# Build the final summary table
summary_rows = []
for i, gene in enumerate(df_rank.index):
    row = {
        'Priority': i + 1,
        'Gene': gene,
        'Composite rank': f"{df_rank.loc[gene, 'composite_rank']:.1f}",
    }
    if 'reversal_score' in df_rank:
        row['Reversal score'] = f"{df_rank.loc[gene, 'reversal_score']:+.4f}"
    if 'specificity_ratio' in df_rank:
        row['Specificity'] = f"{df_rank.loc[gene, 'specificity_ratio']:.2f}"
    if 'mild_shift' in df_rank:
        row['Dose sensitivity'] = f"{df_rank.loc[gene, 'mild_shift']:.4f}"
    row['Clinical status'] = known_targets.get(gene, '—')
    summary_rows.append(row)

df_summary = pd.DataFrame(summary_rows)
print("\n" + "=" * 100)
print("PRIORITIZED DRUG TARGET GENES FOR ALS")
print("=" * 100)
print(df_summary.to_string(index=False))

# Save
df_summary.to_csv('../data/drug_target_ranking.csv', index=False)
df_rank.to_csv('../data/drug_target_ranking_detailed.csv')

In [ ]:
# Final visualization for the slide deck
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Panel A: reversal scores
if 'reversal_score' in df_rank:
    order = df_rank.index
    colors = ['seagreen' if v > 0 else 'indianred' for v in df_rank['reversal_score']]
    axes[0].barh(range(len(order)), df_rank['reversal_score'], color=colors, edgecolor='gray')
    axes[0].set_yticks(range(len(order)))
    axes[0].set_yticklabels([f"#{i+1} {g}" for i, g in enumerate(order)], fontsize=10)
    axes[0].axvline(x=0, color='black', linewidth=0.8)
    axes[0].set_xlabel('Disease reversal score')
    axes[0].set_title('A. Reversal: disease→healthy shift')
    axes[0].invert_yaxis()

# Panel B: specificity
if 'specificity_ratio' in df_rank:
    axes[1].barh(range(len(order)), df_rank.loc[order, 'specificity_ratio'], color='mediumpurple', edgecolor='gray')
    axes[1].set_yticks(range(len(order)))
    axes[1].set_yticklabels([f"#{i+1} {g}" for i, g in enumerate(order)], fontsize=10)
    axes[1].axvline(x=1.0, color='black', linewidth=0.8, linestyle='--')
    axes[1].set_xlabel('Specificity ratio (vulnerable / other)')
    axes[1].set_title('B. Cell-type specificity')
    axes[1].invert_yaxis()

# Panel C: dose sensitivity
if 'mild_shift' in df_rank:
    axes[2].barh(range(len(order)), df_rank.loc[order, 'mild_shift'], color='darkorange', edgecolor='gray')
    axes[2].set_yticks(range(len(order)))
    axes[2].set_yticklabels([f"#{i+1} {g}" for i, g in enumerate(order)], fontsize=10)
    axes[2].set_xlabel('Embedding shift at mild dose')
    axes[2].set_title('C. Dose sensitivity')
    axes[2].invert_yaxis()

plt.suptitle('Drug Target Prioritization — Multi-Criteria Ranking', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../slides/task4_final_ranking.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary

**Drug target prioritization method:**

We combined three independent criteria:
1. **Disease reversal**: projection of perturbation shift onto the disease→healthy direction in GeneFormer embedding space
2. **Cell-type specificity**: ratio of perturbation effect in vulnerable neurons vs other cell types
3. **Dose sensitivity**: magnitude of embedding shift from mild (50%) expression changes

**Validation:**
- Cross-referenced with known ALS therapeutic targets (SOD1/tofersen, ATXN2/ASO trials, STMN2, C9orf72)
- Compared with differential expression ranking between healthy and disease cells
- Pathway enrichment confirms biological relevance

**Limitations:**
- GeneFormer embeddings capture transcriptomic state, not protein-level changes (TDP-43 mislocalization is a protein phenotype)
- In-silico perturbation by expression scaling is an approximation — real gene regulation involves complex feedback loops
- The ranking should be treated as hypothesis-generating, not definitive — experimental validation (e.g., Perturb-seq) would be needed to confirm